# Ноутбук 040: Создание естественных noisy-баз с заменой фрагментов

**Цель:** подготовить отдельные коллекции Qdrant для более реалистичного end-to-end эксперимента по типам шума.

В предыдущей версии noisy-базы создавались как `rag_clean + добавленные шумовые документы`. Для `counterfactual` и `structural` это не всегда реалистично: в базе одновременно оставалась корректная версия знания и её зашумлённая версия.

В этом ноутбуке используется более аккуратная схема:

```text
semantic:       clean-база + дополнительные семантические дистракторы
counterfactual: часть clean-фрагментов заменяется контрафактическими версиями
structural:     часть clean-фрагментов заменяется структурно повреждёнными версиями
```

## Как трактовать уровни 40/80

- Для `semantic` уровень означает долю доступных семантических дистракторов, добавленных в базу.
- Для `counterfactual` и `structural` уровень означает долю доступных единиц замены `(question_id, based_on_title)`, для которых исходные clean-фрагменты удаляются из коллекции и вместо них загружается шумовая версия.

После создания коллекций retrieval будет естественным: `question → одна Qdrant-коллекция → top-k → LLM answer`. Доля шума в retrieved-контексте не задаётся искусственно.

## Шаг 0. Установка зависимостей

In [1]:
!pip install -q qdrant-client fastembed langchain-text-splitters tqdm pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 20.6 MB/s eta 0:00:00


## Шаг 1. Подключение Google Drive и загрузка настроек

Ключи Qdrant лучше хранить в Colab Secrets или переменных окружения: `QDRANT_URL`, `QDRANT_API_KEY`.

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import random
import hashlib
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    from google.colab import userdata
    QDRANT_URL = " "
    QDRANT_API_KEY =" "
except Exception:
    QDRANT_URL = os.environ.get('QDRANT_URL')
    QDRANT_API_KEY = os.environ.get('QDRANT_API_KEY')

assert QDRANT_URL, 'Не найден QDRANT_URL. Укажи его в Colab Secrets или os.environ.'
assert QDRANT_API_KEY, 'Не найден QDRANT_API_KEY. Укажи его в Colab Secrets или os.environ.'

ARTIFACTS_DIR = Path('/content/drive/MyDrive/rag_experiment/artifacts')
assert ARTIFACTS_DIR.exists(), f'Не найдена папка артефактов: {ARTIFACTS_DIR}'

COLLECTION_CLEAN = 'rag_clean'
EMBEDDING_MODEL = 'BAAI/bge-small-en-v1.5'
EMBEDDING_DIM = 384
CHUNK_SIZE = 512
CHUNK_OVERLAP = 64
UPLOAD_BATCH = 256
SCROLL_BATCH = 512

# True удобно при отладке; False — если не хочешь удалять уже созданные коллекции.
RECREATE_COLLECTIONS = True

NOISE_TYPES = ['semantic', 'counterfactual', 'structural']
NOISE_LEVELS = [40, 80]

print(f'Artifacts dir: {ARTIFACTS_DIR}')
print(f'Noise types: {NOISE_TYPES}')
print(f'Noise levels: {NOISE_LEVELS}')

Artifacts dir: /content/drive/MyDrive/rag_experiment/artifacts
Noise types: ['semantic', 'counterfactual', 'structural']
Noise levels: [40, 80]


## Шаг 2. Загрузка артефактов из ноутбуков 01 и 02

In [4]:
with open(ARTIFACTS_DIR / 'questions.json', encoding='utf-8') as f:
    questions = json.load(f)
with open(ARTIFACTS_DIR / 'gold_mapping.json', encoding='utf-8') as f:
    gold_mapping = json.load(f)
with open(ARTIFACTS_DIR / 'noise_cache.json', encoding='utf-8') as f:
    noise_cache = json.load(f)

config_path = ARTIFACTS_DIR / 'config.json'
if config_path.exists():
    with open(config_path, encoding='utf-8') as f:
        base_config = json.load(f)
else:
    base_config = {}

print(f'Вопросов: {len(questions)}')
print(f'Gold mappings: {len(gold_mapping)}')
print(f'Noise cache entries: {len(noise_cache)}')
print('Пример ключей noise_cache:', list(next(iter(noise_cache.values())).keys()))

Вопросов: 100
Gold mappings: 100
Noise cache entries: 100
Пример ключей noise_cache: ['semantic_distractors', 'counterfactuals', 'duplicates', 'structural']


## Шаг 3. Подключение к Qdrant и инициализация embedder

In [5]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, PayloadSchemaType
from fastembed import TextEmbedding
from langchain_text_splitters import RecursiveCharacterTextSplitter

qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=120)

collections = [c.name for c in qdrant.get_collections().collections]
print('Коллекции в Qdrant:')
for c in collections:
    print('  -', c)
assert COLLECTION_CLEAN in collections, f'Не найдена коллекция {COLLECTION_CLEAN}. Сначала выполни ноутбук 01.'

clean_info = qdrant.get_collection(COLLECTION_CLEAN)
print(f'{COLLECTION_CLEAN}: {clean_info.points_count} points')

print('Инициализация FastEmbed...')
embedder = TextEmbedding(model_name=EMBEDDING_MODEL)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', '. ', ' ', ''],
    length_function=len,
)
print('Готово')

Коллекции в Qdrant:
  - rag_clean
  - rag_noisy
rag_clean: 7997 points
Инициализация FastEmbed...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Готово


## Шаг 4. Подготовка документов шума

Здесь формируются два типа объектов:

1. **additive docs** — документы, которые добавляются к базе без удаления clean-версии. Так используется `semantic`.
2. **replacement docs** — документы, которые заменяют исходный clean-фрагмент. Так используются `counterfactual` и `structural`.

Единица замены задаётся парой `(question_id, based_on_title)`. При создании noisy-коллекции все clean-чанки с выбранными `based_on_title` удаляются, а вместо них загружаются соответствующие шумовые версии.

In [6]:
def stable_sample(items, fraction, seed):
    # Детерминированно выбирает fraction элементов из списка.
    items = list(items)
    if not items:
        return []
    if fraction >= 1:
        return items
    rng = random.Random(seed)
    idxs = list(range(len(items)))
    rng.shuffle(idxs)
    k = max(1, int(round(len(items) * fraction)))
    selected = sorted(idxs[:k])
    return [items[i] for i in selected]


def collect_semantic_additive_docs():
    # Семантические дистракторы добавляются как внешние похожие документы.
    docs = []
    for q in questions:
        qid = q['id']
        entry = noise_cache.get(qid, {})
        for i, d in enumerate(entry.get('semantic_distractors', [])):
            docs.append({
                'text': d['text'],
                'title': f"{d['title']} [SEM-{i}]",
                'role': 'injected_semantic_distractor',
                'noise_type': 'semantic',
                'source_question_id': qid,
                'source_question_ids': [qid],
                'based_on_title': d['title'],
                'replacement_mode': 'additive',
            })
    return docs


def collect_replacement_docs(noise_type):
    # Контрафактический и структурный шум заменяют исходные clean-фрагменты.
    assert noise_type in {'counterfactual', 'structural'}
    docs = []
    for q in questions:
        qid = q['id']
        entry = noise_cache.get(qid, {})
        if noise_type == 'counterfactual':
            for i, cf in enumerate(entry.get('counterfactuals', [])):
                if not cf.get('validation_passed'):
                    continue
                based_on_title = cf.get('based_on_title')
                if not based_on_title:
                    continue
                docs.append({
                    'text': cf['text'],
                    'title': f"{based_on_title} [CF-{i}]",
                    'role': 'injected_counterfactual',
                    'noise_type': 'counterfactual',
                    'source_question_id': qid,
                    'source_question_ids': [qid],
                    'based_on_title': based_on_title,
                    'replacement_key': f'{qid}::{based_on_title}',
                    'replacement_mode': 'replace_clean_title',
                    'original_value': cf.get('original_value'),
                    'new_value': cf.get('new_value'),
                })
        elif noise_type == 'structural':
            for i, st in enumerate(entry.get('structural', [])):
                based_on_title = st.get('based_on_title')
                if not based_on_title:
                    continue
                docs.append({
                    'text': st['text'],
                    'title': f"{based_on_title} [STR-{i}]",
                    'role': 'injected_structural',
                    'noise_type': 'structural',
                    'source_question_id': qid,
                    'source_question_ids': [qid],
                    'based_on_title': based_on_title,
                    'replacement_key': f'{qid}::{based_on_title}',
                    'replacement_mode': 'replace_clean_title',
                })
    return docs


semantic_docs = collect_semantic_additive_docs()
replacement_docs = {
    'counterfactual': collect_replacement_docs('counterfactual'),
    'structural': collect_replacement_docs('structural'),
}

print(f'semantic additive docs: {len(semantic_docs)}')
for nt, docs in replacement_docs.items():
    keys = {d['replacement_key'] for d in docs}
    titles = {d['based_on_title'] for d in docs}
    print(f'{nt} replacement docs: {len(docs)}; replacement units: {len(keys)}; titles affected: {len(titles)}')

semantic additive docs: 393
counterfactual replacement docs: 197; replacement units: 197; titles affected: 197
structural replacement docs: 100; replacement units: 100; titles affected: 100


## Шаг 5. Группировка replacement-документов по единицам замены

Если для одного `(question_id, based_on_title)` есть несколько шумовых версий, они рассматриваются как одна единица замены и выбираются вместе.

In [7]:
def group_replacements_by_unit(docs):
    groups = defaultdict(list)
    for d in docs:
        groups[d['replacement_key']].append(d)
    return dict(groups)

replacement_groups = {
    nt: group_replacements_by_unit(docs)
    for nt, docs in replacement_docs.items()
}

for nt, groups in replacement_groups.items():
    sizes = [len(v) for v in groups.values()]
    print(f'{nt}: groups={len(groups)}, docs={sum(sizes)}, docs per group min/max={min(sizes) if sizes else 0}/{max(sizes) if sizes else 0}')

counterfactual: groups=197, docs=197, docs per group min/max=1/1
structural: groups=100, docs=100, docs per group min/max=1/1


## Шаг 6. Служебные функции для создания коллекций

Для `semantic` clean-коллекция копируется целиком, затем добавляются дистракторы. Для `counterfactual` и `structural` clean-коллекция копируется **с исключением заменяемых title**, затем загружаются шумовые версии этих title.

In [8]:
def create_empty_collection(collection_name):
    existing = [c.name for c in qdrant.get_collections().collections]
    if collection_name in existing:
        if RECREATE_COLLECTIONS:
            print(f'Удаляю существующую коллекцию: {collection_name}')
            qdrant.delete_collection(collection_name=collection_name)
        else:
            print(f'Коллекция уже существует, пропускаю создание: {collection_name}')
            return False

    qdrant.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
    )

    for field in ['title', 'role', 'noise_type', 'source_question_ids', 'source_question_id', 'based_on_title', 'replacement_mode']:
        try:
            qdrant.create_payload_index(
                collection_name=collection_name,
                field_name=field,
                field_schema=PayloadSchemaType.KEYWORD,
            )
        except Exception as e:
            print(f'  Индекс {field} не создан/уже существует: {e}')
    return True


def copy_clean_to(collection_name, exclude_titles=None):
    # Копирует rag_clean в новую коллекцию, опционально исключая selected titles.
    exclude_titles = set(exclude_titles or [])
    print(f'Копирование {COLLECTION_CLEAN} → {collection_name}')
    if exclude_titles:
        print(f'  Исключаем clean-title: {len(exclude_titles)}')

    offset = None
    copied = 0
    skipped = 0

    while True:
        points, next_offset = qdrant.scroll(
            collection_name=COLLECTION_CLEAN,
            limit=SCROLL_BATCH,
            offset=offset,
            with_payload=True,
            with_vectors=True,
        )
        if not points:
            break

        batch = []
        for p in points:
            title = p.payload.get('title')
            if title in exclude_titles:
                skipped += 1
                continue
            batch.append(PointStruct(id=p.id, vector=p.vector, payload=p.payload))

        if batch:
            qdrant.upsert(collection_name=collection_name, points=batch, wait=True)
            copied += len(batch)

        if next_offset is None:
            break
        offset = next_offset

    print(f'  Скопировано clean-точек: {copied}; исключено: {skipped}')
    return copied, skipped


def chunk_docs(docs):
    chunks = []
    for doc in docs:
        parts = splitter.split_text(doc['text'])
        for ci, part in enumerate(parts):
            raw_id = f"{doc['noise_type']}::{doc['source_question_id']}::{doc['title']}::{ci}"
            chunk = {
                'chunk_id': hashlib.md5(raw_id.encode('utf-8')).hexdigest(),
                'title': doc['title'],
                'chunk_index': ci,
                'text': part,
                'role': doc['role'],
                'noise_type': doc['noise_type'],
                'source_question_ids': doc.get('source_question_ids', [doc['source_question_id']]),
                'source_question_id': doc['source_question_id'],
                'based_on_title': doc.get('based_on_title'),
                'replacement_mode': doc.get('replacement_mode'),
                'original_value': doc.get('original_value'),
                'new_value': doc.get('new_value'),
            }
            chunks.append({k: v for k, v in chunk.items() if v is not None})
    return chunks


def embed_texts(texts, batch_size=64):
    vectors = []
    for v in tqdm(embedder.embed(texts, batch_size=batch_size), total=len(texts), desc='Embeddings'):
        vectors.append(v.tolist())
    return vectors


def upload_chunks(collection_name, chunks, id_offset):
    if not chunks:
        print('  Нет chunks для загрузки')
        return 0

    vectors = embed_texts([c['text'] for c in chunks])
    points = []
    for idx, (chunk, vec) in enumerate(zip(chunks, vectors)):
        points.append(PointStruct(id=id_offset + idx, vector=vec, payload=chunk))

    for i in tqdm(range(0, len(points), UPLOAD_BATCH), desc=f'Upload {collection_name}'):
        qdrant.upsert(collection_name=collection_name, points=points[i:i + UPLOAD_BATCH], wait=True)

    return len(points)

## Шаг 7. Создание коллекций

Имена коллекций:

```text
rag_noisy_semantic_40
rag_noisy_semantic_80
rag_noisy_counterfactual_replace_40
rag_noisy_counterfactual_replace_80
rag_noisy_structural_replace_40
rag_noisy_structural_replace_80
```

Для совместимости с будущим end-to-end ноутбуком конфигурация будет сохранена в JSON.

In [9]:
collection_report = []

TYPE_OFFSET = {
    'semantic': 1_000_000,
    'counterfactual': 2_000_000,
    'structural': 3_000_000,
}

for noise_type in NOISE_TYPES:
    for level in NOISE_LEVELS:
        fraction = level / 100
        if noise_type == 'semantic':
            collection_name = f'rag_noisy_semantic_{level}'
            design = 'additive_semantic_distractors'
        else:
            collection_name = f'rag_noisy_{noise_type}_replace_{level}'
            design = 'replace_clean_fragments'

        print('\n' + '=' * 90)
        print(f'Создание коллекции: {collection_name}')
        print(f'Тип шума: {noise_type}; уровень: {level}%; схема: {design}')

        should_fill = create_empty_collection(collection_name)
        if not should_fill and not RECREATE_COLLECTIONS:
            info = qdrant.get_collection(collection_name)
            collection_report.append({
                'collection': collection_name,
                'noise_type': noise_type,
                'noise_level': level,
                'design': design,
                'status': 'skipped_existing',
                'points_total': info.points_count,
            })
            continue

        if noise_type == 'semantic':
            selected_docs = stable_sample(semantic_docs, fraction=fraction, seed=SEED + 10_000 + level)
            exclude_titles = set()
            copied, skipped = copy_clean_to(collection_name, exclude_titles=exclude_titles)
            chunks = chunk_docs(selected_docs)
            print(f'  Выбрано semantic-документов для добавления: {len(selected_docs)} из {len(semantic_docs)}')
            print(f'  Semantic chunks после chunking: {len(chunks)}')
        else:
            groups = replacement_groups[noise_type]
            group_items = list(groups.items())
            selected_groups = stable_sample(group_items, fraction=fraction, seed=SEED + hash(noise_type) % 10_000 + level)
            selected_docs = [doc for _, docs in selected_groups for doc in docs]
            exclude_titles = {doc['based_on_title'] for doc in selected_docs if doc.get('based_on_title')}
            copied, skipped = copy_clean_to(collection_name, exclude_titles=exclude_titles)
            chunks = chunk_docs(selected_docs)
            print(f'  Выбрано replacement units: {len(selected_groups)} из {len(group_items)}')
            print(f'  Выбрано replacement docs: {len(selected_docs)}')
            print(f'  Удаляем clean-title: {len(exclude_titles)}')
            print(f'  Replacement chunks после chunking: {len(chunks)}')

        uploaded = upload_chunks(collection_name=collection_name, chunks=chunks, id_offset=TYPE_OFFSET[noise_type] + level * 10_000)
        info = qdrant.get_collection(collection_name)
        collection_report.append({
            'collection': collection_name,
            'noise_type': noise_type,
            'noise_level': level,
            'design': design,
            'status': 'created',
            'clean_points_copied': copied,
            'clean_points_skipped_replaced': skipped,
            'selected_docs': len(selected_docs),
            'selected_chunks_uploaded': uploaded,
            'exclude_titles_count': len(exclude_titles),
            'points_total': info.points_count,
        })
        print(f'  Готово: {collection_name}, всего точек: {info.points_count}')

report_df = pd.DataFrame(collection_report)
report_df


Создание коллекции: rag_noisy_semantic_40
Тип шума: semantic; уровень: 40%; схема: additive_semantic_distractors
Копирование rag_clean → rag_noisy_semantic_40
  Скопировано clean-точек: 7997; исключено: 0
  Выбрано semantic-документов для добавления: 157 из 393
  Semantic chunks после chunking: 263


Embeddings:   0%|          | 0/263 [00:00<?, ?it/s]

Upload rag_noisy_semantic_40:   0%|          | 0/2 [00:00<?, ?it/s]

  Готово: rag_noisy_semantic_40, всего точек: 8260

Создание коллекции: rag_noisy_semantic_80
Тип шума: semantic; уровень: 80%; схема: additive_semantic_distractors
Копирование rag_clean → rag_noisy_semantic_80
  Скопировано clean-точек: 7997; исключено: 0
  Выбрано semantic-документов для добавления: 314 из 393
  Semantic chunks после chunking: 492


Embeddings:   0%|          | 0/492 [00:00<?, ?it/s]

...

  Готово: rag_noisy_structural_replace_40, всего точек: 8004

Создание коллекции: rag_noisy_structural_replace_80
Тип шума: structural; уровень: 80%; схема: replace_clean_fragments
Копирование rag_clean → rag_noisy_structural_replace_80
  Исключаем clean-title: 80
  Скопировано clean-точек: 7893; исключено: 104
  Выбрано replacement units: 80 из 100
  Выбрано replacement docs: 80
  Удаляем clean-title: 80
  Replacement chunks после chunking: 113


Embeddings:   0%|          | 0/113 [00:00<?, ?it/s]

Upload rag_noisy_structural_replace_80:   0%|          | 0/1 [00:00<?, ?it/s]

  Готово: rag_noisy_structural_replace_80, всего точек: 8006


,collection,noise_type,noise_level,design,status,clean_points_copied,clean_points_skipped_replaced,selected_docs,selected_chunks_uploaded,exclude_titles_count,points_total
0,rag_noisy_semantic_40,semantic,40,additive_semantic_distractors,created,7997,0,157,263,0,8260
1,rag_noisy_semantic_80,semantic,80,additive_semantic_distractors,created,7997,0,314,492,0,8489
2,rag_noisy_counterfactual_replace_40,counterfactual,40,replace_clean_fragments,created,7900,97,79,97,79,7997
3,rag_noisy_counterfactual_replace_80,counterfactual,80,replace_clean_fragments,created,7794,203,158,204,158,7998
4,rag_noisy_structural_replace_40,structural,40,replace_clean_fragments,created,7945,52,40,59,40,8004
5,rag_noisy_structural_replace_80,structural,80,replace_clean_fragments,created,7893,104,80,113,80,8006


## Шаг 8. Проверка состава коллекций

Считаем роли документов и убеждаемся, что в `semantic` коллекциях clean-часть сохранена полностью, а в `counterfactual` и `structural` коллекциях часть clean-точек исключена и заменена шумовыми версиями.

In [10]:
def count_by_role(collection_name):
    role_counts = Counter()
    offset = None
    while True:
        points, next_offset = qdrant.scroll(
            collection_name=collection_name,
            limit=SCROLL_BATCH,
            offset=offset,
            with_payload=True,
            with_vectors=False,
        )
        if not points:
            break
        for p in points:
            role_counts[p.payload.get('role', 'unknown')] += 1
        if next_offset is None:
            break
        offset = next_offset
    return dict(role_counts)

role_reports = []
for row in collection_report:
    collection_name = row['collection']
    counts = count_by_role(collection_name)
    role_reports.append({'collection': collection_name, **counts})

role_df = pd.DataFrame(role_reports).fillna(0)
role_df

,collection,gold,distractor,background,injected_semantic_distractor,injected_counterfactual,injected_structural
0,rag_noisy_semantic_40,257,1304,6436,263.0,0.0,0.0
1,rag_noisy_semantic_80,257,1304,6436,492.0,0.0,0.0
2,rag_noisy_counterfactual_replace_40,160,1304,6436,0.0,97.0,0.0
3,rag_noisy_counterfactual_replace_80,54,1304,6436,0.0,204.0,0.0
4,rag_noisy_structural_replace_40,205,1304,6436,0.0,0.0,59.0
5,rag_noisy_structural_replace_80,153,1304,6436,0.0,0.0,113.0


## Шаг 9. Быстрый retrieval sanity-check

Здесь нет принудительного подмешивания шума в top-k. Мы просто проверяем, что поиск работает и смотрим, какие роли попадают в выдачу естественно.

In [11]:
def retrieve(collection_name, query, top_k=10):
    qvec = list(embedder.embed([query]))[0].tolist()
    hits = qdrant.query_points(
        collection_name=collection_name,
        query=qvec,
        limit=top_k,
        with_payload=True,
    ).points
    return hits

rng = random.Random(SEED + 2026)
sample_questions = rng.sample(questions, min(10, len(questions)))

sanity_rows = []
for row in collection_report:
    collection_name = row['collection']
    role_counter = Counter()
    own_noise_hits = 0
    gold_hits = []
    for q in sample_questions:
        hits = retrieve(collection_name, q['question'], top_k=10)
        roles = [h.payload.get('role', 'unknown') for h in hits]
        role_counter.update(roles)
        own_noise_hits += sum(
            1 for h in hits
            if str(h.payload.get('role', '')).startswith('injected_')
            and q['id'] in h.payload.get('source_question_ids', [])
        )
        gold_titles = set(gold_mapping[q['id']]['gold_titles'])
        titles = [h.payload.get('title', '') for h in hits]
        found_gold = sum(1 for g in gold_titles if any(g in t for t in titles))
        gold_hits.append(found_gold / max(1, len(gold_titles)))
    sanity_rows.append({
        'collection': collection_name,
        'noise_type': row['noise_type'],
        'noise_level': row['noise_level'],
        'design': row['design'],
        'avg_gold_recall_at_10_on_sample': float(np.mean(gold_hits)),
        'own_noise_hits_top10_on_sample': own_noise_hits,
        'roles_in_top10_sample': dict(role_counter),
    })

sanity_df = pd.DataFrame(sanity_rows)
sanity_df

,collection,noise_type,noise_level,design,avg_gold_recall_at_10_on_sample,own_noise_hits_top10_on_sample,roles_in_top10_sample
0,rag_noisy_semantic_40,semantic,40,additive_semantic_distractors,0.95,5,"{'gold': 30, 'distractor': 41, 'background': 2..."
1,rag_noisy_semantic_80,semantic,80,additive_semantic_distractors,0.95,12,"{'gold': 27, 'distractor': 37, 'injected_seman..."
2,rag_noisy_counterfactual_replace_40,counterfactual,40,replace_clean_fragments,0.95,4,"{'gold': 24, 'distractor': 44, 'background': 2..."
3,rag_noisy_counterfactual_replace_80,counterfactual,80,replace_clean_fragments,0.95,16,"{'injected_counterfactual': 20, 'distractor': ..."
4,rag_noisy_structural_replace_40,structural,40,replace_clean_fragments,1.00,0,"{'gold': 29, 'distractor': 44, 'background': 27}"
5,rag_noisy_structural_replace_80,structural,80,replace_clean_fragments,1.00,8,"{'gold': 17, 'injected_structural': 10, 'distr..."


## Шаг 10. Сохранение конфигурации для end-to-end генератора

Следующий ноутбук генерации ответов должен выбирать коллекцию так:

- `noise_level = 0` → `rag_clean`;
- `noise_type = semantic`, `noise_level = 40/80` → `rag_noisy_semantic_40/80`;
- `noise_type = counterfactual`, `noise_level = 40/80` → `rag_noisy_counterfactual_replace_40/80`;
- `noise_type = structural`, `noise_level = 40/80` → `rag_noisy_structural_replace_40/80`.

Важно: в end-to-end генераторе больше не нужно вручную смешивать clean/noisy retrieval-кандидатов.

In [12]:
end_to_end_config = {
    'design': 'natural_noisy_collections_add_semantic_replace_counterfactual_structural',
    'description': (
        'Separate Qdrant collections for end-to-end evaluation. Semantic distractors are added as extra documents; '
        'counterfactual and structural noise replace selected clean fragments by title. Retrieval should use one collection per config.'
    ),
    'collection_clean': COLLECTION_CLEAN,
    'noise_types': NOISE_TYPES,
    'noise_levels': NOISE_LEVELS,
    'collections': {'0': COLLECTION_CLEAN},
    'report': collection_report,
    'retrieval_note': 'Use one collection per config; do not manually mix clean and noisy candidates.',
}

for row in collection_report:
    key = f"{row['noise_type']}__{row['noise_level']}"
    end_to_end_config['collections'][key] = row['collection']

out_path = ARTIFACTS_DIR / 'end_to_end_replacement_collections_config.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(end_to_end_config, f, ensure_ascii=False, indent=2)

base_config['end_to_end_replacement_collections'] = end_to_end_config
with open(ARTIFACTS_DIR / 'config.json', 'w', encoding='utf-8') as f:
    json.dump(base_config, f, ensure_ascii=False, indent=2)

print(f'Сохранено: {out_path}')
print('Коллекции для end-to-end:')
for k, v in end_to_end_config['collections'].items():
    print(f'  {k}: {v}')

Сохранено: /content/drive/MyDrive/rag_experiment/artifacts/end_to_end_replacement_collections_config.json
Коллекции для end-to-end:
  0: rag_clean
  semantic__40: rag_noisy_semantic_40
  semantic__80: rag_noisy_semantic_80
  counterfactual__40: rag_noisy_counterfactual_replace_40
  counterfactual__80: rag_noisy_counterfactual_replace_80
  structural__40: rag_noisy_structural_replace_40
  structural__80: rag_noisy_structural_replace_80


## Шаг 11. Итог

После выполнения этого ноутбука можно запускать end-to-end генератор в естественном режиме: `question → collection_by_noise_type_and_level → retrieval → method → generation`.

Состав retrieved-контекста будет определяться только Qdrant retrieval, а не искусственной пропорцией clean/noisy документов.

In [13]:
print('=' * 90)
print('СОЗДАНИЕ ЕСТЕСТВЕННЫХ NOISY-БАЗ С ЗАМЕНОЙ ФРАГМЕНТОВ ЗАВЕРШЕНО')
print('=' * 90)
print('Созданные/проверенные коллекции:')
for row in collection_report:
    print(f"  {row['collection']}: {row.get('points_total', 'unknown')} points; design={row['design']}")
print('Файл конфигурации:')
print(ARTIFACTS_DIR / 'end_to_end_replacement_collections_config.json')

СОЗДАНИЕ ЕСТЕСТВЕННЫХ NOISY-БАЗ С ЗАМЕНОЙ ФРАГМЕНТОВ ЗАВЕРШЕНО
Созданные/проверенные коллекции:
  rag_noisy_semantic_40: 8260 points; design=additive_semantic_distractors
  rag_noisy_semantic_80: 8489 points; design=additive_semantic_distractors
  rag_noisy_counterfactual_replace_40: 7997 points; design=replace_clean_fragments
  rag_noisy_counterfactual_replace_80: 7998 points; design=replace_clean_fragments
  rag_noisy_structural_replace_40: 8004 points; design=replace_clean_fragments
  rag_noisy_structural_replace_80: 8006 points; design=replace_clean_fragments
Файл конфигурации:
/content/drive/MyDrive/rag_experiment/artifacts/end_to_end_replacement_collections_config.json
